# Building CodAdapt

## Goal

Build CodAdapt from scratch to understand how the algorithm works step by step.

CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

This project rebuilds the core ideas from the simplest possible version and gradually introduces bucketization, coded memory, collisions, and adaptive codes.

---

## Version 2

Introduce coded memory.

Instead of using the bucket directly as a memory address, each bucket is mapped to a code.

The model will:

- divide `Age` into buckets
- assign a code to each bucket
- use the code as the memory address
- learn one prediction for each memory cell

In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

## Data preparation

We use `Age` as the only input feature and `Survived` as the target.

Missing ages are filled using the median learned from the training data.

In [3]:
target = "Survived"

age_median = train["Age"].median()

x = train["Age"].fillna(age_median).to_numpy()
y = train[target].to_numpy()

## Building the model

We first divide the continuous `Age` values into four buckets.

Each bucket will point to one value stored in memory.

In [4]:
cuts = np.quantile(x, [0.25, 0.50, 0.75])

bucket = np.digitize(x, cuts)

Each bucket is now mapped to a code.

The code determines the memory address used by the model.

In [5]:
codes = np.array([0, 1, 2, 3])

address = codes[bucket]

In [6]:
memory = np.zeros(4)

In [7]:
for a in range(4):
    memory[a] = y[address == a].mean()

In [8]:
memory

array([0.42647059, 0.40601504, 0.3354232 , 0.4       ])

A prediction is obtained by using the code as an address into memory.

In [9]:
preds = memory[address]

In [10]:
preds[:10]

array([0.40601504, 0.4       , 0.40601504, 0.4       , 0.4       ,
       0.3354232 , 0.4       , 0.42647059, 0.40601504, 0.42647059])

## Prediction and submission

The test ages are converted into buckets using the same cut points learned from the training data.

Each bucket is mapped to a code, which selects the prediction from memory.

In [11]:
x_test = test["Age"].fillna(age_median).to_numpy()

bucket_test = np.digitize(x_test, cuts)
address_test = codes[bucket_test]

probabilities = memory[address_test]
preds = (probabilities >= 0.5).astype(int)

In [12]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": preds
})

submission.to_csv("submission_v2.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Notes

- This second version introduces coded memory.
- `Age` is used as the only input feature.
- Missing `Age` values are filled using the median learned from the training data.
- Continuous values are discretized into four buckets using quantiles.
- Each bucket is mapped to a code.
- The code is used as an address into memory.
- Each memory cell stores the average target value of the samples assigned to that address.
- The same bucket boundaries and codes learned from the training data are reused for the test data.
- The model does not yet use multiple features, feature interactions, collisions, shared multi-resolution encoding, or adaptive codes.
- The final probabilities are converted to `0` or `1` using a threshold of `0.5` for the Titanic submission.
- This notebook is a simplified educational reconstruction of CodAdapt, not the full library implementation.
- CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

Future versions will introduce new components and gradually evolve the architecture.